In [1]:
from pyspark.sql import SparkSession
import getpass

username = getpass.getuser()

In [2]:
spark = SparkSession.builder \
.config("spark.port.ui", 0) \
.config("spark.sql.warehouse.dir", f"/user/{username}/warehouse") \
.enableHiveSupport() \
.master("yarn") \
.getOrCreate()

🏢 The Scenario: Bronze to Silver Pipeline

Building a pipeline for the Analytics team. They need a pristine Silver table that tracks purchased items to build a Star Schema downstream. The raw data arrives as JSON strings.

The Requirements:

    Deduplication: Kafka sometimes sends duplicate messages. Ensure we only process unique events based on the kafka_key.

    Filtering: We only care about events where event_type is "purchase".

    Data Types: The timestamp must be converted to an actual Spark TimestampType.

    Flattening: The analytics team wants a flat table where one row equals one purchased item.
    The final schema must be exactly: event_id (from kafka_key), user_id, event_time, device, item_id, price.

In [3]:
from pyspark.sql.types import *
import pyspark.sql.functions as F

# Generating synthetic raw webhook payloads (Bronze Layer)
data = [
    ('ev_001', '{"user_id": "u123", "event_type": "purchase", "timestamp": "2026-06-01T10:00:00Z", "payload": {"device": "mobile", "items": [{"item_id": "i1", "price": 10.5}, {"item_id": "i2", "price": 20.0}]}}'),
    ('ev_002', '{"user_id": "u456", "event_type": "purchase", "timestamp": "2026-06-01T10:05:00Z", "payload": {"device": "desktop", "items": [{"item_id": "i3", "price": 15.0}]}}'),
    ('ev_003', '{"user_id": "u123", "event_type": "purchase", "timestamp": "2026-06-01T10:10:00Z", "payload": {"device": "mobile", "items": []}}'), # Edge case: empty items
    ('ev_004', '{"user_id": "u789", "event_type": "view", "timestamp": "2026-06-01T10:15:00Z", "payload": {"device": "tablet", "items": null}}'), # Edge case: null items
    ('ev_001', '{"user_id": "u123", "event_type": "purchase", "timestamp": "2026-06-01T10:00:00Z", "payload": {"device": "mobile", "items": [{"item_id": "i1", "price": 10.5}, {"item_id": "i2", "price": 20.0}]}}') # Edge case: Duplicate event from Kafka
]

schema = StructType([
    StructField("kafka_key", StringType(), True),
    StructField("value", StringType(), True) # Raw JSON string
])

bronze_df = spark.createDataFrame(data, schema)
bronze_df.show(truncate=False)

+---------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|kafka_key|value                                                                                                                                                                                             |
+---------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|ev_001   |{"user_id": "u123", "event_type": "purchase", "timestamp": "2026-06-01T10:00:00Z", "payload": {"device": "mobile", "items": [{"item_id": "i1", "price": 10.5}, {"item_id": "i2", "price": 20.0}]}}|
|ev_002   |{"user_id": "u456", "event_type": "purchase", "timestamp": "2026-06-01T10:05:00Z", "payload": {"device": "desktop", "items": [{"item_id": "i3", "price": 15.0}]}}

In [4]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType, ArrayType

streaming_schema = StructType([
        StructField("user_id", StringType()),
        StructField("event_type", StringType()),
        StructField("timestamp", TimestampType()),
        StructField("payload", StructType([
            StructField("device", StringType()),
            StructField("items", ArrayType(
                StructType([
                    StructField("item_id", StringType()),
                    StructField("price", DoubleType())
                ])
            ))
        ]))
    ])

In [5]:
from pyspark.sql import functions as F

parsed_df = bronze_df.withColumn("value", F.from_json(F.col("value"), streaming_schema))

In [6]:
parsed_df.show(truncate=False)

+---------+-------------------------------------------------------------------------+
|kafka_key|value                                                                    |
+---------+-------------------------------------------------------------------------+
|ev_001   |{u123, purchase, 2026-06-01 06:00:00, {mobile, [{i1, 10.5}, {i2, 20.0}]}}|
|ev_002   |{u456, purchase, 2026-06-01 06:05:00, {desktop, [{i3, 15.0}]}}           |
|ev_003   |{u123, purchase, 2026-06-01 06:10:00, {mobile, []}}                      |
|ev_004   |{u789, view, 2026-06-01 06:15:00, {tablet, null}}                        |
|ev_001   |{u123, purchase, 2026-06-01 06:00:00, {mobile, [{i1, 10.5}, {i2, 20.0}]}}|
+---------+-------------------------------------------------------------------------+



In [7]:
parsed_df.printSchema()

root
 |-- kafka_key: string (nullable = true)
 |-- value: struct (nullable = true)
 |    |-- user_id: string (nullable = true)
 |    |-- event_type: string (nullable = true)
 |    |-- timestamp: timestamp (nullable = true)
 |    |-- payload: struct (nullable = true)
 |    |    |-- device: string (nullable = true)
 |    |    |-- items: array (nullable = true)
 |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |-- item_id: string (nullable = true)
 |    |    |    |    |-- price: double (nullable = true)



In [8]:
filtered_df = parsed_df.filter("value.event_type = 'purchase'")
filtered_df.show(truncate=False)

+---------+-------------------------------------------------------------------------+
|kafka_key|value                                                                    |
+---------+-------------------------------------------------------------------------+
|ev_001   |{u123, purchase, 2026-06-01 06:00:00, {mobile, [{i1, 10.5}, {i2, 20.0}]}}|
|ev_002   |{u456, purchase, 2026-06-01 06:05:00, {desktop, [{i3, 15.0}]}}           |
|ev_003   |{u123, purchase, 2026-06-01 06:10:00, {mobile, []}}                      |
|ev_001   |{u123, purchase, 2026-06-01 06:00:00, {mobile, [{i1, 10.5}, {i2, 20.0}]}}|
+---------+-------------------------------------------------------------------------+



In [9]:
from pyspark.sql import Window

window_spec = Window.partitionBy("kafka_key").orderBy(F.desc("value.timestamp"))

dedup_df = filtered_df.withColumn("row_num", F.row_number().over(window_spec)) \
                      .filter(F.col("row_num") == 1) \
                      .drop("row_num")

In [10]:
exploded_df = dedup_df.withColumn("items", F.explode_outer(F.col("value.payload.items")))

In [11]:
exploded_df.show(truncate=False)

+---------+-------------------------------------------------------------------------+----------+
|kafka_key|value                                                                    |items     |
+---------+-------------------------------------------------------------------------+----------+
|ev_001   |{u123, purchase, 2026-06-01 06:00:00, {mobile, [{i1, 10.5}, {i2, 20.0}]}}|{i1, 10.5}|
|ev_001   |{u123, purchase, 2026-06-01 06:00:00, {mobile, [{i1, 10.5}, {i2, 20.0}]}}|{i2, 20.0}|
|ev_003   |{u123, purchase, 2026-06-01 06:10:00, {mobile, []}}                      |null      |
|ev_002   |{u456, purchase, 2026-06-01 06:05:00, {desktop, [{i3, 15.0}]}}           |{i3, 15.0}|
+---------+-------------------------------------------------------------------------+----------+



In [12]:
exploded_df.printSchema()

root
 |-- kafka_key: string (nullable = true)
 |-- value: struct (nullable = true)
 |    |-- user_id: string (nullable = true)
 |    |-- event_type: string (nullable = true)
 |    |-- timestamp: timestamp (nullable = true)
 |    |-- payload: struct (nullable = true)
 |    |    |-- device: string (nullable = true)
 |    |    |-- items: array (nullable = true)
 |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |-- item_id: string (nullable = true)
 |    |    |    |    |-- price: double (nullable = true)
 |-- items: struct (nullable = true)
 |    |-- item_id: string (nullable = true)
 |    |-- price: double (nullable = true)



In [13]:
silver_df = exploded_df.select(
    F.col("kafka_key"),
    F.col("value.user_id").alias("user_id"),
    F.col("value.event_type").alias("event_type"),
    F.col("value.timestamp").alias("event_time"),
    F.col("value.payload.device").alias("device"),
    F.col("items.item_id").alias("item_id"),
    F.col("items.price").alias("item_price")
)

silver_df.show(truncate=False)

+---------+-------+----------+-------------------+-------+-------+----------+
|kafka_key|user_id|event_type|event_time         |device |item_id|item_price|
+---------+-------+----------+-------------------+-------+-------+----------+
|ev_001   |u123   |purchase  |2026-06-01 06:00:00|mobile |i1     |10.5      |
|ev_001   |u123   |purchase  |2026-06-01 06:00:00|mobile |i2     |20.0      |
|ev_003   |u123   |purchase  |2026-06-01 06:10:00|mobile |null   |null      |
|ev_002   |u456   |purchase  |2026-06-01 06:05:00|desktop|i3     |15.0      |
+---------+-------+----------+-------------------+-------+-------+----------+

